<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - PWN, or Binary Exploitation</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 3: Stack Cookies (yum)
</h1>

<h1 style="font-size:30px; font-family:Monaco">1 - First Defense :( </h1>

Unfortunately, hacking is no longer as simple as buffer-overflowing into shellcode and doing whatever you want. In practice, this means that there are more things that need to "align" (i.e. more bad code that needs to be available to us) and that we need to know the techniques for sidestepping these protections.

This part of our introsec series as well as the next will cover two of the most common defenses for the vast majority of architectures and ways to get around them, starting with `Stack Cookies`.

In [ ]:
# create directory for exercise binaries
import os
repo_home = globals()["__vsc_ipynb_file__"]       # get directory of notebook
repo_home = repo_home[:repo_home.rfind('/') + 1]  # remove filename
exercise_home = os.path.join(repo_home, "binaries")

!mkdir -p {exercise_home}
%cd {exercise_home}

<hr style="border:1px solid white">

Try to perform a buffer overflow on the following program like you did during the first two PWN notebooks.

In [ ]:
%%file vuln.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

//gcc vuln.c -o vuln -fstack-protector-all -no-pie -Wno-all

void win(){
    printf("You win!\n");
    system("/bin/sh");
}

int main(void){
    setvbuf(stdout, NULL, _IONBF, 0);
    char buf[24];

    printf("Enter some text: ");
    fgets(buf, 100, stdin);
    printf("You entered: %s\n", buf);
}

In [ ]:
!gcc vuln.c -o vuln -fstack-protector-all -no-pie -Wno-all

In [ ]:
from pwn import *

r = process("./vuln")
r.recvuntil(b"Enter some text: ")

# TODO: enter your payload
r.sendline(b"A" * 100)

print(r.recvall().decode())

What's that?
```
*** stack smashing detected ***: terminated
```
The program detected the overflow and quit. Let's see how it did that.

<h2 style="font-size:20px; font-family:Monaco">1.1 - What are they?</h2>

`Stack Cookies` or `Stack Canaries` are bits of extra data stored in stack frames that are used to detect buffer overflows. These values can be the same every run, but it is much more common that they are random each run.

In [ ]:
%%file examplecookie.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

//gcc examplecookie.c -o examplecookie -fno-stack-protector -no-pie -Wno-all

int __stack_chk = 0xdeadbeef;
int main(void){
    char buf[24];
    int cookie = __stack_chk;

    printf("Enter your favorite kind of cookie: ");
    fgets(buf, 0x24, stdin);
    buf[strcspn(buf, "\n")] = 0;
    printf("\nCookie: 0x%x\n\n", cookie);

    if ( (cookie = cookie ^ __stack_chk) != 0 ){
        printf("Uh oh! Somebody got caught in the cookie jar...\n");
        exit(1);
    }

    printf("Wow! %s is my favorite kind of cookie too!\n", buf);
    return 0;
}

In [ ]:
!gcc examplecookie.c -o examplecookie -fno-stack-protector -no-pie -Wno-all

Try running the program and sending varying input lengths to see what happens.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - How do they work?</h2>

Stack cookies are randomly generated values and as seen in the code above, if you overwrite the cookie, the program terminates early. Ideally, this prevents the function from returning normally and therefore stops any attempts at overwriting the return address.

Below is the typical format of a stack cookie setup:

```c
   0x5555555551e9 <main>:	endbr64
   0x5555555551ed <main+4>:	push   rbp
   0x5555555551ee <main+5>:	mov    rbp,rsp
   0x5555555551f1 <main+8>:	sub    rsp,0x30
   0x5555555551f5 <main+12>:	mov    rax,QWORD PTR fs:0x28
   0x5555555551fe <main+21>:	mov    QWORD PTR [rbp-0x8],rax

After the typical stack frame preservation, we see two new instructions:
1. A `mov` into `RAX` from somewhere labeled `fs:0x28`
2. Putting that value from `fs:0x28` near the top of the stack frame

Where is `fs:0x28` you ask? Well, `fs` is a segment register. Look it up if you are interested but to make a somewhat complicated piece of OS progression history simple, it is used to access `TLS` (thread local storage). When using `gcc` specifically, at an offset of `0x28` from `fs`, a random value generated by the runtime is stored (and typically used as a stack canary).

Now that we have an idea of where it comes from, let's look at the actual verification of the cookie:

```c
   0x5555555552b5 <main+00cc>      mov    rdx, QWORD PTR [rbp-0x8]
   0x5555555552b9 <main+00d0>      sub    rdx, QWORD PTR fs:0x28
   0x5555555552c2 <main+00d9>      je     0x5555555552c9 <main+224>
   0x5555555552c4 <main+00db>      call   0x5555555550b0 <__stack_chk_fail@plt>
   0x5555555552c9 <main+00e0>      leave  
   0x5555555552ca <main+00e1>      ret
```

As we can see above, the cookie is loaded off of the stack into `RDX` and then compared to the original value in `fs`. If they are equal, hop over the `__stack_chk_fail` call and return normally. If someone has overwritten the cookie and the comparison fails, your computer exits with the stack smashing detected warning.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.3 - How can we get around them? </h2>

Simple (but not always easy). We need to figure out what the cookie is!

This is, however, not as easy as simply debugging the program and examining what is stored at `$fs_base+0x28`. Since it is generated by the runtime, this cookie will be different every time you run the program.

Enter: leaks! (sort of)

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Bad plumbing</h1>

Memory leakage is typically problematic in the sense that you will end up using too much (and cause issues for the OS at large) by not marking unused space as available. The kind we are going to talk about is about leaking (or disclosing) information that is stored in memory.

<h2 style="font-size:20px; font-family:Monaco">WARNING!!!! DO NOT BE CONFUSED!!</h2>

Memory leaks in this particular respect are <em>not the same</em> as memory leaks where you forget to free some allocated space. That is a different kind of exploit and one we will talk about in a later notebook. This kind of leak is also known as an information disclosure, where sensitive data not meant for the user to know (e.g. the stack cookie) is given to the user.

There are no memory allocation/freeing shenanigans occurring here, we are just reading out extra data that isn't meant to be available.

If you have questions or are confused, ask!!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 - Leaky Breeki</h2>

In [ ]:
%%file exampleleak.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

//gcc exampleleak.c -o exampleleak -fstack-protector-all -no-pie -Wno-all

int main(void);

void win(){
    printf("You win!\n");
    system("/bin/sh");
}

void foo() {
    setvbuf(stdout, NULL, _IONBF, 0);
    long code = 0xcafebabe;
    char buf[8];
    long *rsp = (long *) &code;
    
    printf("Address of main: %p\n", main);
    printf("Peek at the stack: %lx, %lx, %lx, %lx, %lx, %lx", *rsp, *(rsp + 1), *(rsp + 2), *(rsp + 3), *(rsp + 4), *(rsp + 5));
    printf("\nEnter some text: ");
    fgets(buf, 100, stdin);
}

int main(void){
    foo();
}

In [ ]:
!gcc exampleleak.c -o exampleleak -fstack-protector-all -no-pie

Can you figure out the cookie from the outputs? Now can you exploit this?

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 - It's still just numbers!</h2>

If you read a cookie, you can use your preferred read function from `pwntools` to get the bytes in, slice off any fatty bytes at the beginning/end, and send it back during your ovewrite.

There isn't a whole lot to this! When it comes to challenges (or real-life situations :)) like these, you just need to figure out a way to leak a cookie and then put it back in the right place when overwriting.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Putting it all together:</h1>

Solve [museum](https://labs.rpis.ec/challenges#museum-92) from labs.rpis.ec/intro-pwn.

In [ ]:
%%file securemuseum.c
#include <stdio.h>
#include <stdlib.h>

//gcc securemuseum.c -o securemuseum -no-pie -fstack-protector-all -Wno-all

void win(){
  printf("You win! Here's your shell:\n");
  system("/bin/sh");
}

int get_user_int() {
  int choice = 0;
  scanf("%d", &choice);

  // clean up the input buffer to avoid issues with leftover characters
  int c;
  while ((c = getchar()) != '\n' && c != EOF);

  return choice;
}

int main(void){
  setvbuf(stdout, NULL, _IONBF, 0);

  char name[32];
  unsigned long catalog[5] = { 0x1503, 0x1889, 0x1893, 0x1931, 0x1665 };
  int choice = 0;

  printf("Which exhibit's catalog id would you like (0-4)?: ");
  choice = get_user_int();

  printf("Exhibit #%d has catalog id: 0x%lx\n", choice, catalog[choice]);
  printf("Would you like to sign your name on the guestbook? ");
  gets(name);

  printf("Welcome, %s. Enjoy your visit!\n", name);

  return 0;
}

In [ ]:
!gcc securemuseum.c -o securemuseum -no-pie -fstack-protector-all -Wno-all -g

<h1 style="font-size:30px; font-family:Monaco">Challenge: gallery</h1>

Here is another challenge for you: [gallery](https://labs.rpis.ec/challenges#gallery-93). The source code is provided below for convenience.

In [ ]:
%%file gallery.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>

// gcc gallery.c -o gallery -fstack-protector-all -no-pie -z execstack -Wno-all

int main(void) {
    setvbuf(stdout, NULL, _IONBF, 0);
    setvbuf(stdin,  NULL, _IONBF, 0);

    char name[64];

    printf("=== The Gallery Guestbook ===\n");

    printf("Please sign in for our records: ");
    fgets(name, sizeof(name), stdin);
    name[strcspn(name, "\n")] = 0;

    printf("Now showing the gallery to: ");
    printf(name);
    printf("\n");
    
    printf("Leave a comment in the guestbook: ");
    read(0, name, 200);

    printf("Thanks for visiting!\n");
    return 0;
}

In [ ]:
!gcc gallery.c -o gallery -fstack-protector-all -no-pie -z execstack -Wno-all